# Page 8 — "Best-of-N and Verifiers": the same candidates, judged instead of counted

Page 7 kept the answer that came up most. This page keeps the answer some *rule* likes best, and
then measures what each rule is worth. Nothing is generated: the candidates are page 7's, and
that is the whole point of the comparison. Five selection rules read one pile of chains.

| selector | what it knows | source |
|---|---|---|
| random pick | nothing | the baseline, and equal to single-chain accuracy |
| majority vote | which answer recurs | page 7's rule, unchanged |
| heuristic score | whether the answer is boxed, and how short the text is | ch05 `heuristic_score` |
| average logprob | the model's own confidence in the answer it wrote | ch05 `avg_logprob_answer` |
| oracle | the ground truth | the ceiling, and equal to coverage |

Writes `out/page_08/`: `scores.jsonl`, `derived.json`, `manifest.json`. Its input is
`out/page_07/traces.jsonl`.

The selector ladder and the claims contract live in
[`docs/page_8_refactor.md`](../docs/page_8_refactor.md).

## 0. How to run this

| Mode | What it does | Hardware | Time |
|---|---|---|---|
| **`REPLAY`** *(default)* | every figure from the published candidates and scores | **none** | ~1 min |
| `QUICK` | rescores a reduced pool on your own hardware | ~6 GB GPU | minutes |
| `FULL` | rescores the whole pool | ~6 GB GPU | tens of minutes |

**This notebook generates nothing, but it is not free.** Four of the five selectors are
reductions of page 7's dump and cost nothing at all. `avg_logprob` needs one forward pass per
candidate: no sampling, no generation, but it does need the weights. The spec's claim that this
notebook "runs no generation at all" is right; its implied claim that it therefore needs no model
is not, and the scoring pass below is where the difference lives. `REPLAY` reads the published
`scores.jsonl` instead.

**The LLM judge is scaffolded and off.** [`docs/page_8_refactor.md`](../docs/page_8_refactor.md)
open question 1 leaves it to be decided after the logprob-to-oracle gap is visible. Section 13
prints that gap. Nothing here invents a judge's numbers.

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
jupyter lab
```

Run page 7 first, or drop its published `traces.jsonl` into `out/page_07/`.

# Part 1 — The candidates and the selectors

Loading page 7's pile, defining each rule, and scoring. Nothing here draws anything.

In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "REPLAY"       # "REPLAY" (no GPU), "QUICK" (reduced pool), "FULL"
SEED = 123                # seeds the subsampling below, and nothing else

# Which of page 7's configurations is the pile. None means "whatever page 7 named
# as its pool", which is the allocation optimum's chain length extended to POOL_N.
POOL_CONFIG = None

# The N axis is built by drawing subsets from the pool, so the chain LENGTH is held
# fixed and the only thing changing is how many candidates the rule may look at.
# Anything else would confound "more candidates" with "shorter candidates", which is
# page 7's axis and not this page's.
N_RESAMPLES = 64          # random subsets per (item, N); the sampling error on each point
N_BOOTSTRAP = 1000        # resamples of the ITEM set, which is what the intervals are over

# The buckets are defined by what the vote does, and then the per-bucket curves are
# drawn on the same items. Defining and measuring on the same chains would guarantee
# the shape the page wants, so by default the pile is halved: odd-indexed chains name
# the bucket, even-indexed ones are what the curves are computed from. Set False to
# define and measure on the whole pile, which is the circular version and is only
# useful for showing how much of the effect the circularity was worth.
BUCKET_HOLDOUT = True

QUICK_ITEMS = 25          # QUICK scores this many items and draws every figure from them

# The judge. Built and off: section 6 implements it, section 13 prints the gap that
# decides whether it is worth the half hour, and turning it on is this one flag.
#
# Qwen2.5-7B-Instruct because page 6 already measured it on these same items at 60%
# against the 3B's 43%, so it is a judge whose OWN competence is a published number
# rather than an assumption -- which is exactly what "the judge is a model too" needs.
# 8-bit because 15 GB of bf16 does not fit beside anything on a 16 GB card.
JUDGE_ENABLED = False
JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
JUDGE_QUANT = "8bit"
# The same question put to the model that WROTE the candidates. Nearly free, because
# that checkpoint is already resident for the logprob pass, and it turns "a better
# judge helps" from an assertion into a difference between two measured rungs.
SELF_JUDGE = True

# Where REPLAY reads page 7's candidates and this page's scores.
PAGE_07_TRACES_URL = None
SCORES_URL = None

print(f"mode={RUN_MODE}  resamples={N_RESAMPLES}  judge={'on' if JUDGE_ENABLED else 'off'}")

In [ ]:
import inspect
import json
import math
import random
import textwrap
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import requests
import torch

import harness

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    render_prompt, extract_final_candidate, grade_answer)
from reasoning_from_scratch.ch05 import heuristic_score, avg_logprob_answer

OUT_DIR = Path("out/page_08")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PAGE_07_DIR = Path("out/page_07")

P7_TRACES = PAGE_07_DIR / "traces.jsonl"
P7_DERIVED = PAGE_07_DIR / "derived.json"
SCORES_PATH = OUT_DIR / "scores.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean = harness.mean
median = harness.median

print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

## 1. Page 7's pile

One configuration: one model, one chain length, one temperature, `POOL_N` chains per problem.
Holding all of that fixed is what makes the ladder a fair comparison and what keeps pages 7 and
8 describing one population.

In [ ]:
if PAGE_07_TRACES_URL and not P7_TRACES.exists():
    P7_TRACES.parent.mkdir(parents=True, exist_ok=True)
    print(f"fetching page 7's candidates from {PAGE_07_TRACES_URL}")
    P7_TRACES.write_text(requests.get(PAGE_07_TRACES_URL, timeout=300).text, encoding="utf-8")

assert P7_TRACES.exists(), (
    f"this page consumes page 7's generations and found none at {P7_TRACES}. Run "
    f"page_07_frequency_voting.ipynb, or set PAGE_07_TRACES_URL in the knobs cell.")

# Last write per key wins. Page 7's log is append-only and a configuration that was
# widened from eight chains to sixteen can hold two records under one key; counting one
# chain as two would understate coverage, which is this page's ceiling.
_latest = {}
for _line in P7_TRACES.open(encoding="utf-8"):
    if _line.strip():
        _r = json.loads(_line)
        _latest[(_r["run_id"], _r["config_id"], _r["problem_id"], _r["sample_index"])] = _r
raw = list(_latest.values())
P7_RUN_ID = Counter(r["run_id"] for r in raw).most_common(1)[0][0]
raw = [r for r in raw if r["run_id"] == P7_RUN_ID]
print(f"{len(raw):,} candidate records after collapsing duplicate keys")

P7 = json.loads(P7_DERIVED.read_text()) if P7_DERIVED.exists() else None
if POOL_CONFIG is None:
    assert P7 is not None, (
        "POOL_CONFIG is None and out/page_07/derived.json is not on disk, so there is "
        "nothing to read page 7's pool name from. Set POOL_CONFIG explicitly.")
    POOL_CONFIG = P7["pool"]["config_id"]

MODEL_ID = raw[0]["model_id"]
pool = [r for r in raw if r["config_id"] == POOL_CONFIG and r["model_id"] == MODEL_ID]
assert pool, (f"page 7's traces hold no records for config {POOL_CONFIG!r}. "
              f"Available: {sorted({r['config_id'] for r in raw})}")

POOL = {}
for r in pool:
    POOL.setdefault(r["problem_id"], []).append(r)
for pid in POOL:
    POOL[pid].sort(key=lambda r: r["sample_index"])

# Every item must carry the same number of candidates, or the N axis means a different
# thing on different items and the average across them is not one measurement.
sizes = Counter(len(v) for v in POOL.values())
POOL_N = min(sizes)
if len(sizes) > 1:
    print(f"pool sizes vary {dict(sizes)}; trimming every item to {POOL_N} candidates")
POOL = {p: v[:POOL_N] for p, v in POOL.items()}

if RUN_MODE == "QUICK":
    n_before = len(POOL)
    POOL = {p: POOL[p] for p in sorted(POOL)[:QUICK_ITEMS]}
    print(f"QUICK: {len(POOL)} of {n_before} items")

ITEMS = sorted(POOL)
T = POOL[ITEMS[0]][0]
print(f"page 7 run   {P7_RUN_ID}")
print(f"model        {MODEL_ID}")
print(f"config       {POOL_CONFIG}   T={T['temperature']}  {T['chain_budget']} tokens per chain")
print(f"pile         {len(ITEMS)} problems x {POOL_N} candidates = "
      f"{len(ITEMS) * POOL_N:,} chains, none of them generated here")
print(f"per candidate: {mean([r['own_tokens'] for v in POOL.values() for r in v]):,.0f} tokens "
      f"of thinking on average, {mean([r['truncated'] for v in POOL.values() for r in v]):.0%} "
      f"truncated and forced")

## 2. The vote, carried over unchanged

Page 7's tally is one of the rungs, so it is reimplemented here rather than approximated: the
same `number_only` vote key, the same equivalence relation, the same rule that a tie is not a
vote. If this drifts from page 7, the two pages report different numbers for the same procedure,
so section 13 checks the overlap against page 7's own `derived.json`.

In [ ]:
NO_ANSWER = None


def vote_key(text):
    cand = extract_final_candidate(text, fallback="number_only")
    return cand if cand and cand.strip() else NO_ANSWER


_EQ = {}


def equivalent(a, b):
    if a == b:
        return True
    k = (a, b) if a <= b else (b, a)
    if k not in _EQ:
        try:
            _EQ[k] = bool(grade_answer(a, b)) or bool(grade_answer(b, a))
        except Exception:
            _EQ[k] = False
    return _EQ[k]


def cluster(keys):
    labels, order = [], []
    for k in keys:
        if k is NO_ANSWER:
            labels.append(NO_ANSWER)
            continue
        hit = next((lab for lab in order if equivalent(lab, k)), None)
        if hit is None:
            order.append(k)
            hit = k
        labels.append(hit)
    return labels, order


def plurality(cands):
    """The class the most candidates agree on, or None on a tie or an empty tally."""
    labels, _ = cluster([vote_key(c["trace"]) for c in cands])
    counts = Counter([l for l in labels if l is not NO_ANSWER])
    mc = counts.most_common()
    if not mc:
        return None
    top = mc[0][1]
    winners = [a for a, c in mc if c == top]
    return winners[0] if len(winners) == 1 else None


print("the vote on the full pile, as a sanity read before anything is judged:")
_v = mean([bool(plurality(POOL[p]) is not None
                and grade_answer(plurality(POOL[p]), POOL[p][0]["ground_truth"]))
           for p in ITEMS])
_c = mean([any(r["correct"] for r in POOL[p]) for p in ITEMS])
print(f"  majority at N={POOL_N}   {_v:>6.0%}")
print(f"  coverage             {_c:>6.0%}   the ceiling every selector on this page shares")
print(f"  the gap              {_c - _v:>+6.0%}   which is this page's y-axis")

## 3. The heuristic scorer, printed rather than described

ch05's `heuristic_score` is the judge that grades presentation. That is not an analogy imposed
on it: it rewards a boxed final answer, adds a bonus that decays with the length of the text, and
knows nothing whatsoever about whether the answer is right.

The weights are what make it that judge, so they are printed here rather than cited.

In [ ]:
print(inspect.getsource(heuristic_score))
print("defaults:", {k: v.default for k, v in
                    inspect.signature(heuristic_score).parameters.items()
                    if v.default is not inspect.Parameter.empty})

demo = [
    (r"\boxed{7}", "short and boxed"),
    ("Let me work through this carefully. " * 40 + r"\boxed{7}", "long and boxed"),
    ("The answer is 7.", "short, not boxed"),
]
print("\nwhat it rewards, on text whose correctness it cannot see:")
for text, label in demo:
    print(f"  {heuristic_score(text):>6.3f}   {label:<20} ({len(text)} chars)")

# What it reduces to on THIS pile, which is the thing the page will say out loud.
lens = [len(r["trace"]) for v in POOL.values() for r in v]
boxed = mean(["\\boxed{" in r["trace"] for v in POOL.values() for r in v])
scores = [heuristic_score(r["trace"]) for v in POOL.values() for r in v]
HEURISTIC_IS_LENGTH = boxed > 0.95
print(f"\non this pile: {boxed:.0%} of candidates carry a box, traces run "
      f"{min(lens):,}-{max(lens):,} chars")
print(f"  score range {min(scores):.3f} to {max(scores):.3f}, of which the boxed bonus is a "
      f"constant 2.000")
if HEURISTIC_IS_LENGTH:
    print("\n  Every candidate is boxed, so the boxed bonus cancels and the rule ranks by the")
    print("  brevity term alone. On a pile of full-length chains 'heuristic score' means")
    print("  SHORTEST WINS and nothing else. The page should say that, because it is a")
    print("  sharper version of 'grades on presentation' than the phrase itself.")
else:
    print(f"\n  {1 - boxed:.0%} of candidates carry no box, so the boxed bonus is still doing")
    print("  work here and the rule is not purely a length ranking.")

## 4. The logprob scorer needs the weights, not a generation

ch05's `avg_logprob_answer` averages the model's log-probability over the answer tokens, given
everything before them. One forward pass per candidate, no sampling and no generation.

Two adapters stand between it and a 🤗 checkpoint, and they are here rather than in a rewritten
copy of the function so that the scorer stays the book's:

- the book's model returns a logits tensor and a transformers model returns a `ModelOutput`;
- the book's tokenizer encodes without special tokens, and `AutoTokenizer.encode` adds them by
  default, which would put a marker inside the sequence being scored.

**What counts as "the answer" is a choice, so both readings are measured.**

- **the span** — the tokens from the last `\boxed{` to the end, conditioned on everything before
  them. This is the spec's reading and the one plotted: it asks how confident the model is in the
  answer it landed on. It averages over five to fifteen tokens, so it is noisy, and it favours
  short common answers.
- **the whole chain** — every token the model wrote, conditioned on the prompt alone. This is the
  usual "sequence confidence", and it carries a length bias instead.

Neither is obviously right. The span is plotted, the chain is reported in section 7b, and the
rank correlation between them says how much the choice mattered.

In [ ]:
class _LogitsOnly:
    """A transformers model that answers like the book's model does."""

    def __init__(self, model):
        self.m = model

    def __call__(self, ids):
        return self.m(ids).logits


class _PlainEncode:
    """A tokenizer whose `.encode` adds nothing of its own."""

    def __init__(self, tok):
        self.t = tok

    def encode(self, s):
        return self.t(s, add_special_tokens=False).input_ids


def answer_span(trace):
    """(everything before the answer, the answer). Falls back to the whole trace when
    a candidate never opened a box, which is the case the vote calls 'no answer'."""
    i = trace.rfind("\\boxed{")
    return (trace[:i], trace[i:]) if i != -1 else ("", trace)


print("span split, on one real candidate:")
_r = POOL[ITEMS[0]][0]
_head, _tail = answer_span(_r["trace"])
print(f"  conditioning ends: …{_head[-70:]!r}")
print(f"  scored tokens:      {_tail[:70]!r}")

# ---------------------------------------------------------------------------
# The judge asks a different question of the same machinery, so it is defined
# here beside the other scorer rather than in its own section: put the candidate
# to a model and read how much it prefers "Yes" to "No". One forward pass, no
# generation, and no verdict to parse -- a free-text score would fail on the
# candidates where the model hedges, which are exactly the interesting ones.
# ---------------------------------------------------------------------------
JUDGE_PROMPT = (
    "You are grading a proposed solution to a mathematics problem.\n\n"
    "Problem:\n{question}\n\n"
    "Proposed solution:\n{solution}\n\n"
    "Is the final answer correct? Reply with one word: Yes or No."
)
_YES_NO = {}


def yes_no_ids(tok):
    """The single tokens the judge's answer would start with. Asserted single, because
    scoring the first token of a two-token word would score a prefix and not a verdict."""
    key = id(tok)
    if key not in _YES_NO:
        ids = {}
        for word in ("Yes", "No"):
            enc = tok(word, add_special_tokens=False).input_ids
            assert len(enc) == 1, (
                f"{word!r} is {len(enc)} tokens on this tokenizer, so the judge cannot be "
                f"read off one position. Pick different verdict words and record them.")
            ids[word] = enc[0]
        _YES_NO[key] = ids
    return _YES_NO[key]


@torch.inference_mode()
def yes_no_score(model, tok, question, solution):
    """log P(Yes) - log P(No) at the judge's first answer position. Positive means the
    judge believes the solution; the magnitude is what ranks candidates. It never sees
    the ground truth, which is what keeps it below the oracle rung."""
    ids = yes_no_ids(tok)
    chat = tok.apply_chat_template(
        [{"role": "user", "content": JUDGE_PROMPT.format(question=question,
                                                         solution=solution)}],
        tokenize=False, add_generation_prompt=True)
    enc = tok(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
    logits = model(**enc).logits[0, -1]
    lp = torch.log_softmax(logits.float(), dim=-1)
    return (lp[ids["Yes"]] - lp[ids["No"]]).item()

## 5. The scoring pass

One forward pass per candidate, resumable, written to `scores.jsonl`. `REPLAY` reads the
published file instead and loads nothing.

In [ ]:
SCORE_ID = f"p08-{P7_RUN_ID}-{POOL_CONFIG}"
SCORES = {}

if RUN_MODE == "REPLAY":
    if SCORES_URL and not SCORES_PATH.exists():
        print(f"fetching published scores from {SCORES_URL}")
        SCORES_PATH.write_text(requests.get(SCORES_URL, timeout=300).text, encoding="utf-8")
    assert SCORES_PATH.exists(), (
        f"REPLAY needs published scores at {SCORES_PATH} and found none. Set SCORES_URL in "
        f"the knobs cell, or use RUN_MODE = 'QUICK' to score a reduced pool yourself.")
    for line in SCORES_PATH.open(encoding="utf-8"):
        if line.strip():
            s = json.loads(line)
            if s["score_id"] == SCORE_ID:
                SCORES[(s["problem_id"], s["sample_index"])] = s
    missing = [(p, r["sample_index"]) for p in ITEMS for r in POOL[p]
               if (p, r["sample_index"]) not in SCORES]
    assert not missing, (
        f"the published scores miss {len(missing)} of {len(ITEMS) * POOL_N} candidates; "
        f"that file was made against a different pool.")
    print(f"replaying {len(SCORES):,} scores, nothing loaded")
else:
    device = get_device()
    got = harness.load_model(MODEL_ID, device=device, dtype=harness.pick_dtype(device),
                             warmup_tokens=8)
    model, tokenizer = got["model"], got["tokenizer"]
    scorer_model, scorer_tok = _LogitsOnly(model), _PlainEncode(tokenizer)

    store = harness.TraceStore(SCORES_PATH, SCORE_ID, ("problem_id", "sample_index"))
    store.report()

    bar = harness.progress(ITEMS, desc="scoring", unit="item")
    for pid in bar:
        chat = tokenizer.apply_chat_template(
            [{"role": "user", "content": render_prompt(POOL[pid][0]["question"])}],
            tokenize=False, add_generation_prompt=True)
        for r in POOL[pid]:
            hit = store.get(pid, r["sample_index"])
            if hit is not None:
                SCORES[(pid, r["sample_index"])] = hit
                continue
            head, tail = answer_span(r["trace"])
            # Two calls, not one: ch05's function owns the forward pass, and calling it
            # twice keeps the scorer the book's rather than a reimplementation of it that
            # shares a pass. It costs a second cheap pass per candidate.
            lp_span = avg_logprob_answer(scorer_model, scorer_tok, chat + head, tail,
                                         device=model.device) if tail.strip() else None
            lp_chain = avg_logprob_answer(scorer_model, scorer_tok, chat, r["trace"],
                                          device=model.device) if r["trace"].strip() else None
            rec = {
                "score_id": SCORE_ID, "problem_id": pid, "sample_index": r["sample_index"],
                "avg_logprob": lp_span,          # the span, and the one plotted
                "avg_logprob_chain": lp_chain,   # the whole chain, reported in 7b
                "heuristic": heuristic_score(r["trace"]),
                "n_chars": len(r["trace"]), "answer_chars": len(tail),
                "scorer": "reasoning_from_scratch.ch05.avg_logprob_answer",
            }
            if SELF_JUDGE:
                rec["judge_self"] = yes_no_score(model, tokenizer, r["question"], r["trace"])
            SCORES[(pid, r["sample_index"])] = store.append(rec)
    del model, got, scorer_model
    harness.free_memory()
    print(f"\n{len(SCORES):,} candidates scored")

# The heuristic is pure text analysis, so it is recomputed here in every mode: it costs
# nothing and it means a replay is not trusting a published number it could derive.
for pid in ITEMS:
    for r in POOL[pid]:
        s = SCORES[(pid, r["sample_index"])]
        s["heuristic"] = heuristic_score(r["trace"])

lp_missing = sum(1 for s in SCORES.values() if s["avg_logprob"] is None)
print(f"candidates with no scorable answer span: {lp_missing} of {len(SCORES)}")

## 6. The judge, built and off

A second model reads each candidate and says whether it believes it, using the same Yes/No
machinery as section 4. Two of them, and the pair is the point:

- **`judge_self`** — the checkpoint that *wrote* the candidates, grading its own work. Free,
  because it is already resident for the logprob pass, and it is the control: a model asked to
  judge what it just produced.
- **`judge`** — a stronger checkpoint. `Qwen2.5-7B-Instruct` by default, because page 6 already
  measured it on these same items at 60% against the 3B's 43%, so the page can say what the
  judge's own competence is instead of assuming it. Off by default; section 13 prints the gap
  that decides whether it earns its half hour.

**Same family is a real limitation and the page must carry it.** A 7B Qwen shares the 3B's
pretraining and probably a lot of its error modes, so it is likely to be fooled by the same
traps. A judge from another family would be a stronger test of "a better judge helps"; this one
is a cheaper test of "a bigger judge helps". If the measured reliability comes out close to the
3B's, that ambiguity is the reason to reach for another family before concluding anything.

In [ ]:
JUDGE_SCORED = 0
if JUDGE_ENABLED and RUN_MODE != "REPLAY":
    assert JUDGE_MODEL, "JUDGE_ENABLED is on but JUDGE_MODEL is unset"
    print(f"loading the judge: {JUDGE_MODEL} at {JUDGE_QUANT}")
    jdev = get_device()
    jgot = harness.load_model(JUDGE_MODEL, device=jdev, dtype=harness.pick_dtype(jdev),
                              load_in_8bit=(JUDGE_QUANT == "8bit"), warmup_tokens=8)
    jmodel, jtok = jgot["model"], jgot["tokenizer"]
    print("verdict token ids:", yes_no_ids(jtok))

    jstore = harness.TraceStore(SCORES_PATH, SCORE_ID + "-judge",
                               ("problem_id", "sample_index"))
    bar = harness.progress(ITEMS, desc="judging", unit="item")
    for pid in bar:
        for r in POOL[pid]:
            hit = jstore.get(pid, r["sample_index"])
            if hit is None:
                hit = jstore.append({"score_id": SCORE_ID + "-judge", "problem_id": pid,
                                     "sample_index": r["sample_index"],
                                     "judge": yes_no_score(jmodel, jtok, r["question"],
                                                           r["trace"]),
                                     "judge_model": JUDGE_MODEL, "judge_quant": JUDGE_QUANT})
                JUDGE_SCORED += 1
            SCORES[(pid, r["sample_index"])]["judge"] = hit["judge"]
    del jmodel, jgot
    harness.free_memory()
    print(f"{JUDGE_SCORED:,} candidates judged by {JUDGE_MODEL}")
elif JUDGE_ENABLED:
    # A replay of a run that had a judge reads its verdicts back with everything else.
    for line in SCORES_PATH.open(encoding="utf-8"):
        if line.strip():
            s = json.loads(line)
            if s.get("score_id") == SCORE_ID + "-judge":
                k = (s["problem_id"], s["sample_index"])
                if k in SCORES:
                    SCORES[k]["judge"] = s["judge"]
                    JUDGE_SCORED += 1
    assert JUDGE_SCORED, ("JUDGE_ENABLED is on but the published scores hold no judge "
                          "verdicts; that run was made with the judge off")
    print(f"replayed {JUDGE_SCORED:,} judge verdicts")

HAVE_JUDGE = JUDGE_ENABLED and all("judge" in SCORES[(p, r["sample_index"])]
                                   for p in ITEMS for r in POOL[p])
HAVE_SELF_JUDGE = all("judge_self" in SCORES[(p, r["sample_index"])]
                      for p in ITEMS for r in POOL[p])
print(f"\njudge: {'on, ' + str(JUDGE_MODEL) if HAVE_JUDGE else 'off'}   "
      f"self-judge: {'on' if HAVE_SELF_JUDGE else 'off'}")
if not HAVE_JUDGE:
    print("The ladder stands on the free rules plus the oracle ceiling. Section 13 prints")
    print("the headroom a judge would have to fill, which is what decides whether to run it.")

## 7. The rules

Each takes a list of candidates and returns the index of the one it picks. That signature is
the page's whole argument: **same candidates, same compute, five answers.**

**Ties among scores break toward the lowest sample index**, which is arbitrary and stated rather
than hidden. The majority rule's ties are not broken at all, because page 7 does not break them.

In [ ]:
def sel_random(cands, rng):
    return rng.randrange(len(cands))


def sel_majority(cands, rng):
    """Page 7's rule. Returns the index of a candidate carrying the winning answer, or
    None when the vote ties or nobody answered: a non-vote is not a pick."""
    win = plurality(cands)
    if win is None:
        return None
    for i, c in enumerate(cands):
        k = vote_key(c["trace"])
        if k is not NO_ANSWER and equivalent(k, win):
            return i
    return None


def sel_heuristic(cands, rng):
    return max(range(len(cands)),
               key=lambda i: (SCORES[(cands[i]["problem_id"], cands[i]["sample_index"])]
                              ["heuristic"], -i))


def sel_logprob(cands, rng):
    def key(i):
        s = SCORES[(cands[i]["problem_id"], cands[i]["sample_index"])]["avg_logprob"]
        return (-math.inf if s is None else s, -i)
    return max(range(len(cands)), key=key)


def sel_oracle(cands, rng):
    return next((i for i, c in enumerate(cands) if c["correct"]), None)


def by_score(field):
    """A selector that takes the highest score in `field`, ties to the lowest index."""
    def pick(cands, rng):
        def key(i):
            s = SCORES[(cands[i]["problem_id"], cands[i]["sample_index"])].get(field)
            return (-math.inf if s is None else s, -i)
        return max(range(len(cands)), key=key)
    return pick


SELECTORS = [
    ("random", "picks one at random", sel_random),
    ("majority", "keeps the answer that recurs", sel_majority),
    ("heuristic", "keeps the shortest cleanly boxed answer", sel_heuristic),
    ("logprob", "keeps the answer the model is most confident in", sel_logprob),
]
if HAVE_SELF_JUDGE:
    SELECTORS.append(("judge-self", "asks the model that wrote it whether it believes it",
                      by_score("judge_self")))
if HAVE_JUDGE:
    SELECTORS.append((f"judge", f"asks {JUDGE_MODEL.split('/')[-1]} whether it believes it",
                      by_score("judge")))
SELECTORS.append(("oracle", "keeps a correct one if the pile holds any", sel_oracle))
print(f"{len(SELECTORS)} rules, all reading the same {len(ITEMS) * POOL_N:,} candidates:")
for name, what, _ in SELECTORS:
    print(f"  {name:<11}{what}")

## 7b. What the two scorers actually rank by

Before either is called a judge, it is worth measuring what it is doing, because both turn out to
be proxies for something plainer than their names suggest.

`heuristic_score` ranks by length, which section 3 already showed. `avg_logprob` ranks by how
close a chain is to the one the model would have written greedily, and page 7 generated exactly
that chain at temperature 0 on the same items, so the agreement is measurable rather than
arguable. **A logprob selector that just recovers the greedy chain is not worthless, but it is a
different claim from "the model knows which of its answers is right", and the page must make the
one the data supports.**

In [ ]:
# Does the highest-logprob candidate agree with the length ranking?
rank_agree = []
for pid in ITEMS:
    c = POOL[pid]
    by_lp = sorted(range(len(c)), key=lambda i: -(SCORES[(pid, c[i]["sample_index"])]
                                                  ["avg_logprob"] or -9e9))
    by_len = sorted(range(len(c)), key=lambda i: len(c[i]["trace"]))
    rank_agree.append(by_lp[0] == by_len[0])
print(f"the logprob pick is also the shortest candidate on {mean(rank_agree):.0%} of items")

# Does it agree with the greedy chain page 7 generated at temperature 0?
GREEDY_AGREEMENT = None
greedy_cfg = next((c for c in {r["config_id"] for r in raw}
                   if c.startswith("t0.00-") and c.endswith(POOL_CONFIG.split("-")[-1])), None)
if greedy_cfg:
    greedy = {}
    for r in raw:
        if r["config_id"] == greedy_cfg and r["sample_index"] == 0:
            greedy[r["problem_id"]] = r
    shared = [p for p in ITEMS if p in greedy]
    if shared:
        agree, gcorrect = [], []
        for pid in shared:
            c = POOL[pid]
            pick = c[sel_logprob(c, None)]
            gk, pk = vote_key(greedy[pid]["trace"]), vote_key(pick["trace"])
            agree.append(gk is not None and pk is not None and equivalent(gk, pk))
            gcorrect.append(greedy[pid]["correct"])
        GREEDY_AGREEMENT = dict(config_id=greedy_cfg, n_items=len(shared),
                                answer_agreement=mean(agree),
                                greedy_accuracy=mean(gcorrect))
        print(f"\nthe greedy chain page 7 generated at {greedy_cfg}, on the same {len(shared)} "
              f"items:")
        print(f"  the logprob pick gives the same answer as greedy on "
              f"{GREEDY_AGREEMENT['answer_agreement']:.0%} of items")
        print(f"  greedy alone solves {GREEDY_AGREEMENT['greedy_accuracy']:.0%}")
        if GREEDY_AGREEMENT["answer_agreement"] > 0.8:
            print("\n  Read that carefully. At this rate the logprob rung is largely recovering")
            print("  the greedy chain, so any gain it shows over one sampled chain is mostly")
            print("  'greedy beats a sample', not 'the model can tell which answer is right'.")
            print("  The page may not call it a confidence signal without saying this.")
else:
    print("\nno temperature-0 chains in page 7's file at this cap; the greedy comparison is "
          "skipped")

# The logprob variant that is measured and NOT plotted: the whole chain rather than the
# answer span. Reported here so the choice between them is visible rather than silent.
sel_chain = by_score("avg_logprob_chain")
agree, chain_hits, span_hits = [], [], []
for pid in ITEMS:
    c = POOL[pid]
    i_span, i_chain = sel_logprob(c, None), sel_chain(c, None)
    agree.append(i_span == i_chain)
    span_hits.append(c[i_span]["correct"])
    chain_hits.append(c[i_chain]["correct"])
LOGPROB_VARIANTS = dict(
    span=dict(what="tokens from the last \\boxed{ to the end, conditioned on the rest",
              best_of_n=mean(span_hits), plotted=True),
    chain=dict(what="every token the model wrote, conditioned on the prompt alone",
               best_of_n=mean(chain_hits), plotted=False),
    same_pick_rate=mean(agree))
print(f"\nthe two readings of 'the model's confidence', at N={POOL_N}:")
print(f"  {'variant':<10}{'best-of-N':>11}   what it averages over")
for k in ("span", "chain"):
    v = LOGPROB_VARIANTS[k]
    print(f"  {k:<10}{v['best_of_n']:>11.0%}   {v['what']}"
          + ("   <- plotted" if v["plotted"] else ""))
print(f"  they pick the same candidate on {LOGPROB_VARIANTS['same_pick_rate']:.0%} of items")
if abs(LOGPROB_VARIANTS["span"]["best_of_n"] - LOGPROB_VARIANTS["chain"]["best_of_n"]) > 0.05:
    print("\n  The two readings differ by more than five points, so which one the page plots")
    print("  is a choice that changes the figure. Say which, and why, in the caption.")

## 8. The N axis, built by subsampling

The pile holds `POOL_N` candidates per problem. A point at N is the average over
`N_RESAMPLES` random subsets of that size, drawn without replacement with a seeded generator, so
the x-axis is *how many candidates the rule may look at* with chain length held fixed.

Every rule sees the **same subsets**, drawn once per (item, N, repeat) and handed to all five.
Comparing rules across different draws would put sampling noise into the spread that the figure
reads as a difference between rules.

In [ ]:
N_AXIS = [n for n in (1, 2, 4, 8, 16, 32, 64) if n <= POOL_N]
if N_AXIS[-1] != POOL_N:
    N_AXIS.append(POOL_N)


def subsets_for(pid, n, repeats, size=None):
    """The draws every selector sees at this (item, N). Exhaustive when the pile is
    small enough that resampling would just repeat the same subsets."""
    size = len(POOL[pid]) if size is None else size
    if n >= size:
        return [list(range(size))]
    rng = random.Random(harness.sha256_of([str(SEED), pid, str(n), str(size)])[:15])
    return [rng.sample(range(size), n) for _ in range(repeats)]


def cluster_ci(per_item, reps=N_BOOTSTRAP, tag=""):
    """A 95% interval over ITEMS, which are the independent unit here.

    The draws inside one item are subsets of one pile and are heavily correlated, so a
    Wilson interval on the draw count would treat 6,400 correlated draws as 6,400
    independent trials and come out roughly eight times too narrow. Resampling items
    with replacement keeps the unit of independence honest.
    """
    vals = list(per_item)
    if not vals:
        return (0.0, 0.0)
    rng = random.Random(harness.sha256_of([str(SEED), "ci", tag, str(len(vals))])[:15])
    n, means = len(vals), []
    for _ in range(reps):
        means.append(sum(vals[rng.randrange(n)] for _ in range(n)) / n)
    means.sort()
    return (means[int(0.025 * reps)], means[int(0.975 * reps)])


def run_ladder(item_ids, pool_for, n_axis, tag=""):
    """Every selector over the same draws, at every N. `pool_for` returns the candidate
    list an item contributes, so the buckets below can hand it a held-out half."""
    out = {name: [] for name, _, _ in SELECTORS}
    picks = []
    for n in n_axis:
        per_item = {name: [] for name, _, _ in SELECTORS}
        for pid in item_ids:
            cands_all = pool_for(pid)
            if len(cands_all) < 1:
                continue
            hits = {name: [] for name, _, _ in SELECTORS}
            for rep, idx in enumerate(subsets_for(pid, n, N_RESAMPLES, len(cands_all))):
                cands = [cands_all[i] for i in idx]
                rng = random.Random(
                    harness.sha256_of([str(SEED), tag, pid, str(n), str(rep)])[:15])
                for name, _, fn in SELECTORS:
                    pick = fn(cands, rng)
                    ok = bool(pick is not None and cands[pick]["correct"])
                    hits[name].append(ok)
                    if n == n_axis[-1] and not tag:
                        picks.append(dict(
                            problem_id=pid, selector=name, n=n,
                            pick=None if pick is None else cands[pick]["sample_index"],
                            correct=ok))
            for name, _, _ in SELECTORS:
                per_item[name].append(mean(hits[name]))
        for name, _, _ in SELECTORS:
            vals = per_item[name]
            lo, hi = cluster_ci(vals, tag=f"{tag}{name}{n}")
            out[name].append(dict(n=n, accuracy=mean(vals), ci_lo=lo, ci_hi=hi,
                                  n_items=len(vals), n_draws=len(vals) * N_RESAMPLES))
    return out, picks


LADDER, PICKS = run_ladder(ITEMS, lambda pid: POOL[pid], N_AXIS)

print(f"  {'N':>4}" + "".join(f"{name:>12}" for name, _, _ in SELECTORS))
for i, n in enumerate(N_AXIS):
    print(f"  {n:>4}" + "".join(f"{LADDER[name][i]['accuracy']:>11.0%} "
                                for name, _, _ in SELECTORS))
print(f"\nwith 95% intervals over the {len(ITEMS)} items, at N={N_AXIS[-1]}:")
for name, _, _ in SELECTORS:
    p = LADDER[name][-1]
    print(f"  {name:<12}{p['accuracy']:>7.0%}   {p['ci_lo']:.0%} to {p['ci_hi']:.0%}")
print(f"\n{N_RESAMPLES} draws per item per N, except where the subset is the whole pile.")
print("The intervals are over items, not over draws: the draws inside one item are")
print("subsets of one pile and are not independent of each other.")

# A consistency check that costs nothing: picking at random must land on the same
# number as one chain, because it IS one chain. If it does not, the draws are wrong.
single = mean([mean([c["correct"] for c in POOL[pid]]) for pid in ITEMS])
rand_at_1 = LADDER["random"][0]["accuracy"]
# Three standard errors on the draw count, floored, because the check has to survive a
# QUICK run with a tenth of the draws without either passing vacuously or failing.
TOL = max(0.03, 3 * 0.5 / max(len(ITEMS) * N_RESAMPLES, 1) ** 0.5)
print(f"\nrandom at N=1 {rand_at_1:.1%} against the mean single chain {single:.1%} "
      f"(gap {abs(rand_at_1 - single) * 100:.1f} pts, tolerance {TOL * 100:.1f})")
assert abs(rand_at_1 - single) < TOL, (
    "picking at random from one candidate is not the same as taking that candidate; "
    "the subsampling is not doing what it says")

# Part 2 — What the page shows

## 9. Figure 1 — the selector ladder

Accuracy against N, one line per rule, all reading the same candidates. **The figure's job is
the vertical spread at fixed N, not the climb.** Same pile, same compute, five answers, and the
distance between the top and the bottom line is entirely the selection rule.

In [ ]:
import matplotlib.pyplot as plt

INK = {"random": "#c4c0b8", "majority": "#3b4a6b", "heuristic": "#b23",
       "logprob": "#a2704a", "oracle": "#4a7a5c"}

fig, ax = plt.subplots(figsize=(8.4, 4.6))
for name, _, _ in SELECTORS:
    pts = LADDER[name]
    ax.plot([p["n"] for p in pts], [p["accuracy"] * 100 for p in pts],
            marker="o", ms=4, lw=2 if name != "oracle" else 1.6,
            ls="--" if name == "oracle" else "-",
            color=INK[name], label=name)

top = LADDER["oracle"][-1]["accuracy"] * 100
bot = LADDER["random"][-1]["accuracy"] * 100
ax.annotate("", xy=(N_AXIS[-1], top), xytext=(N_AXIS[-1], bot),
            arrowprops=dict(arrowstyle="<->", color="#555", lw=1))
ax.text(N_AXIS[-1] * 0.96, (top + bot) / 2,
        f"{top - bot:.0f} points already\ngenerated and\nthrown away",
        ha="right", va="center", fontsize=8, color="#555")

ax.set_xscale("log", base=2)
ax.set_xticks(N_AXIS)
ax.set_xticklabels([str(n) for n in N_AXIS])
ax.set_xlabel(f"candidates the rule may look at ({T['chain_budget']} tokens each, T={T['temperature']})")
ax.set_ylabel("share of problems solved (%)")
ax.set_ylim(0, min(100, top + 12))
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=2)
ax.set_title(f"{MODEL_ID.split('/')[-1]}, n={len(ITEMS)} problems, one pile of "
             f"{POOL_N} candidates each", fontsize=9, loc="left", color="#555")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_1_ladder.png", dpi=160)
plt.show()

SPREAD = {n: dict(n=n,
                  top=max(LADDER[s][i]["accuracy"] for s, _, _ in SELECTORS if s != "oracle"),
                  bottom=min(LADDER[s][i]["accuracy"] for s, _, _ in SELECTORS),
                  oracle=LADDER["oracle"][i]["accuracy"])
          for i, n in enumerate(N_AXIS)}
for n in N_AXIS:
    s = SPREAD[n]
    print(f"  N={n:<3} best rule {s['top']:.0%}, worst {s['bottom']:.0%}, "
          f"ceiling {s['oracle']:.0%}   spread {100 * (s['top'] - s['bottom']):.0f} pts")

## 10. Figure 2 — the three buckets

Every item falls into one of three, by what page 7's candidates did:

- **solvable** — the modal answer is correct. Page 7's regime, and voting works.
- **trap** — the modal answer is wrong and a correct answer appears at least once. The only
  place selection can beat counting.
- **hopeless** — no candidate is correct. The coverage ceiling, where no rule helps and only
  more generation or a better model can.

Bucketing costs no inference; it is a reduction of the pile. **The base rate is the page's most
quotable number**, and the trap bucket's size is the gate on the whole second beat.

In [ ]:
# Which chains name the bucket, and which are left to draw the curves from. Splitting
# them is the difference between a finding and a tautology: define the trap bucket on
# the chains you then measure and majority accuracy is guaranteed to be low there,
# because that is how the bucket was chosen.
NAMING = {pid: POOL[pid][0::2] for pid in ITEMS} if BUCKET_HOLDOUT else \
    {pid: POOL[pid] for pid in ITEMS}
MEASURING = {pid: POOL[pid][1::2] for pid in ITEMS} if BUCKET_HOLDOUT else \
    {pid: POOL[pid] for pid in ITEMS}
BUCKET_N = min(len(v) for v in MEASURING.values())
BUCKET_AXIS = [n for n in N_AXIS if n <= BUCKET_N] or [BUCKET_N]


def bucket_of(cands):
    win = plurality(cands)
    has = any(c["correct"] for c in cands)
    if win is not None and grade_answer(win, cands[0]["ground_truth"]):
        return "solvable"
    return "trap" if has else "hopeless"


BUCKET = {pid: bucket_of(NAMING[pid]) for pid in ITEMS}
BASE_RATE = {b: mean([BUCKET[p] == b for p in ITEMS]) for b in ("solvable", "trap", "hopeless")}
FULL_PILE_RATE = {b: mean([bucket_of(POOL[p]) == b for p in ITEMS])
                  for b in ("solvable", "trap", "hopeless")}

print(f"named by {len(next(iter(NAMING.values())))} chains, measured on "
      f"{BUCKET_N}, n={len(ITEMS)} problems"
      + ("" if BUCKET_HOLDOUT else "   (BUCKET_HOLDOUT off: same chains do both)"))
for b, share in BASE_RATE.items():
    lo, hi = harness.wilson(round(share * len(ITEMS)), len(ITEMS))
    print(f"  {b:<10}{share:>6.0%}  {lo:.0%}-{hi:.0%}   {'#' * round(share * 34)}")
print(f"\n  for the record, the same three shares over the whole pile of {POOL_N}: "
      + ", ".join(f"{b} {v:.0%}" for b, v in FULL_PILE_RATE.items()))
print("  The quotable base rate is the first table. The second is what the page would")
print("  have reported if it had defined and measured on the same chains.")

# The trap bucket is page 7's handoff number by construction: an item is a trap exactly
# when a correct answer is in the pile and the vote did not return it.
print(f"\n  trap = 'the right answer was there and the vote threw it away', which is the")
print(f"  number page 7 closes on. The two pages are quoting one measurement.")

TRAP_OK = BASE_RATE["trap"] >= 0.08
if not TRAP_OK:
    print(f"\n!! The trap bucket is {BASE_RATE['trap']:.0%} of items, which is too thin to plot")
    print("   a curve on. docs/page_8_refactor.md open question 3 lists the options: widen the")
    print("   item set, lower N (traps are more visible at small N), or fall back on a")
    print("   perturbed attractor family. Beat 2 does not ship as specced until this passes.")

# Per bucket, the same ladder, drawn from the chains that did NOT name the bucket.
BUCKET_LADDER = {}
for b in ("solvable", "trap", "hopeless"):
    ids = [p for p in ITEMS if BUCKET[p] == b]
    if not ids:
        continue
    lad, _ = run_ladder(ids, lambda pid: MEASURING[pid], BUCKET_AXIS, tag=f"bucket-{b}-")
    BUCKET_LADDER[b] = dict(n_items=len(ids), share=len(ids) / len(ITEMS),
                            points=[dict(n=BUCKET_AXIS[i],
                                         **{name: lad[name][i]["accuracy"]
                                            for name, _, _ in SELECTORS})
                                    for i in range(len(BUCKET_AXIS))],
                            ladder={name: lad[name] for name, _, _ in SELECTORS})

for b, d in BUCKET_LADDER.items():
    print(f"\n{b}  ({d['n_items']} items, {d['share']:.0%})")
    print(f"  {'N':>4}" + "".join(f"{name:>12}" for name, _, _ in SELECTORS))
    for p in d["points"]:
        print(f"  {p['n']:>4}" + "".join(f"{p[name]:>11.0%} " for name, _, _ in SELECTORS))

MAJORITY_DECLINES_ON_TRAPS = None
if "trap" in BUCKET_LADDER and len(BUCKET_AXIS) > 1:
    pts = BUCKET_LADDER["trap"]["points"]
    lad = BUCKET_LADDER["trap"]["ladder"]["majority"]
    MAJORITY_DECLINES_ON_TRAPS = (pts[-1]["majority"] < pts[0]["majority"]
                                  and lad[-1]["ci_hi"] < lad[0]["ci_lo"])
    print(f"\non the trap slice, majority goes {pts[0]['majority']:.0%} "
          f"[{lad[0]['ci_lo']:.0%}-{lad[0]['ci_hi']:.0%}] -> {pts[-1]['majority']:.0%} "
          f"[{lad[-1]['ci_lo']:.0%}-{lad[-1]['ci_hi']:.0%}]")
    print(f"-> {'declines, and the intervals do not overlap' if MAJORITY_DECLINES_ON_TRAPS else 'not resolved: the intervals overlap'}")
    print(f"   ({BUCKET_LADDER['trap']['n_items']} trap items is a small slice, so this is the "
          f"least resolved claim on the page.)")

In [ ]:
fig, ax0 = plt.subplots(figsize=(7.6, 2.1))
left = 0.0
for b, colour in (("solvable", "#3b4a6b"), ("trap", "#b23"), ("hopeless", "#c4c0b8")):
    ax0.barh([0], [BASE_RATE[b] * 100], left=left * 100, color=colour, height=0.5)
    if BASE_RATE[b] > 0.06:
        ax0.text((left + BASE_RATE[b] / 2) * 100, 0,
                 f"{b}\n{BASE_RATE[b]:.0%}", ha="center", va="center", fontsize=9,
                 color="#fff" if b != "hopeless" else "#333")
    left += BASE_RATE[b]
ax0.set_xlim(0, 100)
ax0.set_ylim(-0.6, 0.6)
ax0.set_yticks([])
ax0.set_xlabel("share of problems (%)")
ax0.set_title(f"what the vote does, over {len(ITEMS)} problems", fontsize=9, loc="left",
              color="#555")
ax0.spines[["top", "right", "left"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2_buckets.png", dpi=160)
plt.show()

## 11. Figure 3 — the trap slice, on its own

The one place selection can beat counting, given a figure to itself because it is the page's
proof and because it is the least resolved thing on the page. Three or four small multiples
would hide both facts.

**The bands are what to read.** They are bootstrap intervals over the items in this slice, and
the slice is small, so they are wide. If the majority line's decline does not clear them, the
page says the decline is not resolved rather than pointing at a slope.

The chains drawn here did not name the bucket; the other half of the pile did. Without that
split the decline would be guaranteed by the definition rather than measured.

In [ ]:
if "trap" in BUCKET_LADDER:
    d = BUCKET_LADDER["trap"]
    fig, ax = plt.subplots(figsize=(7.6, 4.4))
    for name, _, _ in SELECTORS:
        pts = d["ladder"][name]
        xs = [p["n"] for p in pts]
        ax.plot(xs, [p["accuracy"] * 100 for p in pts], marker="o", ms=4,
                lw=2 if name == "majority" else 1.5,
                ls="--" if name == "oracle" else "-",
                color=INK.get(name, "#777"), label=name, zorder=3)
        if name in ("majority", "oracle"):
            ax.fill_between(xs, [p["ci_lo"] * 100 for p in pts],
                            [p["ci_hi"] * 100 for p in pts],
                            color=INK.get(name, "#777"), alpha=0.12, lw=0)
    ax.set_xscale("log", base=2)
    ax.set_xticks(BUCKET_AXIS)
    ax.set_xticklabels([str(n) for n in BUCKET_AXIS])
    ax.set_xlabel("candidates the rule may look at")
    ax.set_ylabel("share of the trap slice solved (%)")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=2)
    ax.set_title(f"the trap slice: {d['n_items']} of {len(ITEMS)} problems "
                 f"({d['share']:.0%}) where the vote is wrong and a correct answer is "
                 f"in the pile", fontsize=9, loc="left", color="#555")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure_3_traps.png", dpi=160)
    plt.show()
    print(f"bands are 95% bootstrap intervals over the {d['n_items']} items in the slice.")
else:
    print("no trap items; figure 3 has nothing to draw and derived.json says so")

## 12. Figure 4 — verifier quality, measured

**Selector reliability: on items where a correct candidate exists, how often does this rule pick
one?** One number per rule, computed on the same pile, and it replaces the reliability slider the
current page asks the reader to drag.

The crossing with the majority-vote reference line is the page's second half, and it is a
measured crossing rather than a modelled one: below that reliability, judging is worse than
counting and page 7's tool keeps its place in the toolbox.

In [ ]:
reachable = [p for p in ITEMS if any(c["correct"] for c in POOL[p])]
RELIABILITY = []
for name, what, _ in SELECTORS:
    picks = [q for q in PICKS if q["selector"] == name and q["problem_id"] in set(reachable)]
    rel = mean([q["correct"] for q in picks]) if picks else 0.0
    acc = LADDER[name][-1]["accuracy"]
    RELIABILITY.append(dict(selector=name, what=what, reliability=rel,
                            best_of_n_accuracy=acc, n=POOL_N))

print(f"on the {len(reachable)} of {len(ITEMS)} problems where a correct candidate exists:")
print(f"  {'selector':<12}{'reliability':>13}{'best-of-N':>12}")
for r in RELIABILITY:
    print(f"  {r['selector']:<12}{r['reliability']:>13.0%}{r['best_of_n_accuracy']:>12.0%}")

maj = next(r for r in RELIABILITY if r["selector"] == "majority")
below = [r["selector"] for r in RELIABILITY
         if r["best_of_n_accuracy"] < maj["best_of_n_accuracy"] and r["selector"] != "random"]
print(f"\nrules that do WORSE than counting: {', '.join(below) if below else 'none'}")
JUDGING_CAN_LOSE = bool(below)

fig, ax = plt.subplots(figsize=(7.2, 4.2))
for r in RELIABILITY:
    ax.scatter(r["reliability"] * 100, r["best_of_n_accuracy"] * 100, s=54,
               color=INK[r["selector"]], zorder=3)
    ax.annotate(r["selector"], (r["reliability"] * 100, r["best_of_n_accuracy"] * 100),
                textcoords="offset points", xytext=(8, -3), fontsize=9, color="#333")
ax.axhline(maj["best_of_n_accuracy"] * 100, ls="--", lw=1, color="#3b4a6b")
ax.text(2, maj["best_of_n_accuracy"] * 100 + 1.2, "majority vote", fontsize=8, color="#3b4a6b")
ax.set_xlabel("measured selector reliability: picks a correct candidate when one exists (%)")
ax.set_ylabel(f"best-of-{POOL_N} accuracy (%)")
ax.set_xlim(0, 105)
ax.set_ylim(0, min(100, max(r["best_of_n_accuracy"] for r in RELIABILITY) * 100 + 10))
ax.grid(alpha=0.2)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_4_reliability.png", dpi=160)
plt.show()

## 13. The worked case, and whether a judge is worth building

One problem where the heuristic scorer picked a short, cleanly boxed, wrong answer over a longer
correct one. That is the judge grading on presentation, shown rather than asserted, and it is the
single most persuasive artifact this page can carry.

The gap between logprob and oracle is printed underneath, because
[`docs/page_8_refactor.md`](../docs/page_8_refactor.md) open question 1 says to decide the judge
after seeing it.

In [ ]:
WORKED = None
cases = []
for pid in ITEMS:
    cands = POOL[pid]
    h = sel_heuristic(cands, None)
    if cands[h]["correct"]:
        continue
    right = [c for c in cands if c["correct"]]
    if not right:
        continue
    longest = max(right, key=lambda c: len(c["trace"]))
    if len(longest["trace"]) <= len(cands[h]["trace"]):
        continue          # want the picture the page describes: short wrong over long right
    cases.append((len(longest["trace"]) - len(cands[h]["trace"]), pid, h,
                  longest["sample_index"]))
cases.sort(reverse=True)

if cases:
    _, pid, hi, ri = cases[0]
    picked = POOL[pid][hi]
    passed = next(c for c in POOL[pid] if c["sample_index"] == ri)
    sp, sr = SCORES[(pid, hi)], SCORES[(pid, ri)]
    WORKED = dict(problem_id=pid, question=picked["question"],
                  ground_truth=picked["ground_truth"],
                  n_cases=len(cases),
                  picked=dict(sample_index=hi, answer=picked["extracted_answer"],
                              correct=False, chars=len(picked["trace"]),
                              heuristic=sp["heuristic"], avg_logprob=sp["avg_logprob"],
                              trace=picked["trace"]),
                  passed_over=dict(sample_index=ri, answer=passed["extracted_answer"],
                                   correct=True, chars=len(passed["trace"]),
                                   heuristic=sr["heuristic"], avg_logprob=sr["avg_logprob"],
                                   trace=passed["trace"]))
    print(textwrap.fill("Q: " + picked["question"], 92))
    print("ground truth:", picked["ground_truth"])
    print(f"\n{len(cases)} problems show this shape; the widest is below.\n")
    for label, c, s in (("the heuristic picked", picked, sp),
                        ("it passed over", passed, sr)):
        print("=" * 96)
        print(f"{label}: chain {c['sample_index']}   {len(c['trace'])} chars   "
              f"heuristic {s['heuristic']:.3f}   logprob "
              f"{'n/a' if s['avg_logprob'] is None else format(s['avg_logprob'], '.3f')}   "
              f"answer {c['extracted_answer']!r}   "
              f"{'CORRECT' if c['correct'] else 'wrong'}")
        print(textwrap.fill(c["trace"].replace("\n", " "), 92,
                            initial_indent="    ", subsequent_indent="    ")[:1400])
        print()
    print("The rule rewarded a box and shortness. Neither is evidence about the answer.")
else:
    print("no problem where the heuristic picked a short wrong answer over a longer right one;")
    print("figure 4's worked case has no specimen and derived.json flags it")

lp = next(r for r in RELIABILITY if r["selector"] == "logprob")
orc = next(r for r in RELIABILITY if r["selector"] == "oracle")
GAP_PTS = (orc["best_of_n_accuracy"] - lp["best_of_n_accuracy"]) * 100
print(f"\nlogprob {lp['best_of_n_accuracy']:.0%}, oracle {orc['best_of_n_accuracy']:.0%}: "
      f"{GAP_PTS:.0f} points of headroom a better judge could take.")
print("Over roughly ten points is worth building a judge for; under it, the four free rules")
print("already say what the page needs to say. That is open question 1, answered by the run.")

## 14. Agreement with page 7

The majority rung at page 7's own N must reproduce page 7's number, because it is the same rule
on the same candidates. A disagreement means one of the two implementations drifted, and it would
show up on the page as two numbers for one procedure.

In [ ]:
PAGE_7_AGREEMENT = None
if P7 is not None:
    p7_pool = P7["handoff"]
    ours = LADDER["majority"][-1]["accuracy"]
    cov_ours = LADDER["oracle"][-1]["accuracy"]
    PAGE_7_AGREEMENT = dict(
        config_id=POOL_CONFIG, n=POOL_N,
        page_07_vote=p7_pool["plurality_accuracy"], page_08_vote=ours,
        page_07_coverage=p7_pool["coverage"], page_08_coverage=cov_ours,
        vote_delta_pts=(ours - p7_pool["plurality_accuracy"]) * 100,
        coverage_delta_pts=(cov_ours - p7_pool["coverage"]) * 100,
        same_item_count=(len(ITEMS) == P7["manifest"]["items"]["n"]))
    print(f"  {'':<14}{'page 7':>10}{'page 8':>10}{'delta':>9}")
    print(f"  {'the vote':<14}{p7_pool['plurality_accuracy']:>10.1%}{ours:>10.1%}"
          f"{PAGE_7_AGREEMENT['vote_delta_pts']:>+9.1f}")
    print(f"  {'coverage':<14}{p7_pool['coverage']:>10.1%}{cov_ours:>10.1%}"
          f"{PAGE_7_AGREEMENT['coverage_delta_pts']:>+9.1f}")
    if not PAGE_7_AGREEMENT["same_item_count"]:
        print(f"\n  (this notebook is reading {len(ITEMS)} items against page 7's "
              f"{P7['manifest']['items']['n']}; in QUICK that is expected)")
    elif abs(PAGE_7_AGREEMENT["vote_delta_pts"]) > 0.01:
        print("\n!! The same rule on the same candidates gave two answers. One of the two vote")
        print("   implementations has drifted; fix it before either page ships.")
else:
    print("out/page_07/derived.json not found; the cross-check is skipped")

## 15. Artifacts

`derived.json` is the contract with the page. The candidates themselves are page 7's download,
so this page bundles the aggregates, the buckets, the reliability table and the worked case, and
links page 7's `traces.jsonl` for everything else.

In [ ]:
manifest = {
    "score_id": SCORE_ID,
    "notebook": "page_08_best_of_n.ipynb",
    "notebook_sha256": harness.file_sha256("page_08_best_of_n.ipynb"),
    "harness": harness.fingerprint(),
    "page": 8, "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": harness.git_sha(),
    "generation": "none: every candidate is page 7's",
    "candidates": {"source": "out/page_07/traces.jsonl", "page_07_run_id": P7_RUN_ID,
                   "config_id": POOL_CONFIG, "model_id": MODEL_ID,
                   "temperature": T["temperature"], "top_p": T.get("top_p"),
                   "chain_budget": T["chain_budget"], "pool_n": POOL_N,
                   "n_items": len(ITEMS)},
    "selectors": {
        "random": {"cost": "free", "params": {}},
        "majority": {"cost": "free",
                     "rule": "plurality over ch03-equivalent answer classes; a tie is not a vote",
                     "params": {}},
        "heuristic": {"cost": "free",
                      "source": "reasoning_from_scratch.ch05.heuristic_score",
                      "params": {k: v.default for k, v in
                                 inspect.signature(heuristic_score).parameters.items()
                                 if v.default is not inspect.Parameter.empty}},
        "logprob": {"cost": "two forward passes per candidate",
                    "source": "reasoning_from_scratch.ch05.avg_logprob_answer",
                    "scored_span": "from the last \\boxed{ to the end, conditioned on the rest",
                    "variants": LOGPROB_VARIANTS},
        "oracle": {"cost": "free", "rule": "ground truth; the coverage ceiling"},
        "judge_self": {"built": True, "enabled": HAVE_SELF_JUDGE, "model": MODEL_ID,
                       "rule": "log P(Yes) - log P(No) at the first answer position",
                       "prompt": JUDGE_PROMPT},
        "judge": {"built": True, "enabled": HAVE_JUDGE, "model": JUDGE_MODEL,
                  "quant": JUDGE_QUANT if HAVE_JUDGE else None,
                  "rule": "log P(Yes) - log P(No) at the first answer position",
                  "prompt": JUDGE_PROMPT, "n_scored": JUDGE_SCORED,
                  "same_family_as_the_writer": True,
                  "why_off": None if HAVE_JUDGE else
                             "docs/page_8_refactor.md open question 1; decided after the "
                             "logprob-to-oracle gap in section 13"},
    },
    "subsampling": {"n_axis": N_AXIS, "n_resamples": N_RESAMPLES, "seed": SEED,
                    "rule": "subsets drawn without replacement, identical across selectors",
                    "score_tie_break": "lowest sample index",
                    "intervals": f"cluster bootstrap over items, {N_BOOTSTRAP} resamples; "
                                 f"the draws inside an item are not independent"},
    "buckets": {"holdout": BUCKET_HOLDOUT, "naming_chains": POOL_N - BUCKET_N
                if BUCKET_HOLDOUT else POOL_N, "measuring_chains": BUCKET_N,
                "why": "defining a bucket on the chains you then measure guarantees the "
                       "shape; the pile is halved so it does not",
                "base_rate_if_circular": FULL_PILE_RATE},
    "what_the_scorers_rank_by": {
        "heuristic_is_a_length_ranking": HEURISTIC_IS_LENGTH,
        "logprob_vs_greedy": GREEDY_AGREEMENT},
    "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                "grade": "reasoning_from_scratch.ch03.grade_answer"},
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__},
}

derived = {
    "manifest": manifest,
    # Read before any number.
    "claims": {
        "selection_spread_exists": (SPREAD[N_AXIS[-1]]["top"]
                                    > SPREAD[N_AXIS[-1]]["bottom"] + 0.02),
        "trap_bucket_plottable": TRAP_OK,
        "majority_declines_on_traps": bool(MAJORITY_DECLINES_ON_TRAPS),
        "judging_can_lose_to_counting": JUDGING_CAN_LOSE,
        "worked_case_exists": WORKED is not None,
        # False means the logprob rung is mostly recovering the greedy chain, and the
        # page may not describe it as the model knowing which answer is right.
        "logprob_is_more_than_greedy": (GREEDY_AGREEMENT is None
                                        or GREEDY_AGREEMENT["answer_agreement"] <= 0.8),
        "buckets_are_held_out": BUCKET_HOLDOUT,
        "agrees_with_page_7": (PAGE_7_AGREEMENT is None
                               or not PAGE_7_AGREEMENT["same_item_count"]
                               or abs(PAGE_7_AGREEMENT["vote_delta_pts"]) <= 0.01),
        "judge_built": HAVE_JUDGE,
    },
    "figure_1": {"n_axis": N_AXIS,
                 "ladder": [{"selector": name, "what": what, "points": LADDER[name]}
                            for name, what, _ in SELECTORS],
                 "spread": [SPREAD[n] for n in N_AXIS]},
    "figure_2": {"base_rate": BASE_RATE,
                 "definitions": {
                     "solvable": "the modal answer is correct",
                     "trap": "the modal answer is wrong and a correct answer appears at least once",
                     "hopeless": "no candidate is correct"},
                 "base_rate_if_circular": FULL_PILE_RATE,
                 "held_out": BUCKET_HOLDOUT,
                 "buckets": BUCKET_LADDER},
    # The trap slice has its own figure: it is the page's proof and its least resolved
    # claim, and small multiples would hide both.
    "figure_3": {"bucket": "trap", "axis": BUCKET_AXIS,
                 "slice": BUCKET_LADDER.get("trap"),
                 "majority_declines": MAJORITY_DECLINES_ON_TRAPS},
    "figure_4": {"reliability": RELIABILITY,
                 "majority_reference": maj["best_of_n_accuracy"],
                 "worse_than_counting": below,
                 "logprob_to_oracle_gap_pts": GAP_PTS,
                 "worked_case": WORKED},
    "page_7_agreement": PAGE_7_AGREEMENT,
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"derived.json   {harness.kb(DERIVED_PATH):>10,.1f} KB   <- bundled with the page")
print(f"manifest.json  {harness.kb(MANIFEST_PATH):>10,.1f} KB   <- bundled with the page")
print(f"scores.jsonl   {harness.kb(SCORES_PATH):>10,.1f} KB   <- published as a download")
print("the candidates themselves are page 7's traces.jsonl, linked rather than copied")

blocking = ["selection_spread_exists", "agrees_with_page_7"]
failed = [k for k in blocking if not derived["claims"][k]]
print(f"\n{'TRANSLATABLE' if not failed else 'NOT TRANSLATABLE: ' + ', '.join(failed)}")
for k, v in derived["claims"].items():
    print(f"   {'ok ' if v else 'NO '} {k}")
print("\nThe non-blocking gates change what the page SAYS. `trap_bucket_plottable` false means")
print("beat 2 does not get its per-bucket curves; `worked_case_exists` false means figure 4")
print("loses the specimen beside it. See docs/page_8_refactor.md open questions 2 and 3.")

## 16. Notes for re-derivation

- Nothing here is generated. Every number is a reduction of page 7's pile, plus one forward pass
  per candidate for the logprob rung. Re-running the ladder after changing a selector costs
  seconds.
- The N axis is subsampled from a fixed-length pool on purpose. Page 7's N axis changes the
  chain length as it changes N, because its budget is matched; this page's does not, because
  otherwise the ladder would be measuring page 7's variable.
- `random` is a rule with variance, not a formula. Its line is the average over the same draws
  every other rule sees, and at `N_RESAMPLES` draws per item it should sit within a point or two
  of the single-chain accuracy page 7 reports.
- Reliability is defined only on items where a correct candidate exists. On hopeless items every
  rule is wrong by construction, and including them would compress the differences the figure
  exists to show.
- **The judge is two rungs, not one.** `judge-self` is the checkpoint that wrote the candidates
  grading its own work, and `judge` is a bigger one grading them. Both read log P(Yes) minus
  log P(No) at one position: no generation, no verdict to parse, and a continuous score that
  still ranks when the model hedges. The default judge is the same family as the writer, so a
  reliability close to the self-judge's is ambiguous between "a bigger judge does not help" and
  "a judge with the same training does not help", and settling that needs another family.
- **Two readings of the logprob rung are measured and one is plotted.** The span is plotted; the
  whole chain is in `selectors.logprob.variants`, along with how often the two pick the same
  candidate. If they diverge, the caption has to name which one it is showing.
- The heuristic's weights are ch05's defaults and are recorded in the manifest. Changing them
  changes what the page's "grades on presentation" sentence means, so change them in the knobs
  and say so, or not at all.
- Ties among scores break toward the lowest sample index. The majority rule's ties are not
  broken, because page 7 does not break them.

What the page may claim is [`docs/page_8_refactor.md`](../docs/page_8_refactor.md), checked at
translation ([`docs/page_6_refactor.md`](../docs/page_6_refactor.md) section 6).